### Definition

**Adam** is an adaptive learning rate optimization algorithm that calculates individual learning rates for different parameters by combining two concepts: **Momentum** (tracking the mean of past gradients) and **RMSprop** (tracking the uncentered variance of past gradients).

**AdamW** is a direct mathematical fix to Adam that properly decouples weight decay (L2 regularization) from the gradient update step, preventing regularization from being incorrectly scaled by historical gradient magnitudes.

---

### How Adam Works: The Two "Moments"

To understand why Adam is so powerful, you have to look at how it keeps track of two rolling averages (called "moments") of the gradients during training:

1. **First Moment ($m_t$ - The Mean / Momentum):**
* This tracks the *direction* of the gradients over time.
* Just like a heavy boulder rolling down a hill, momentum helps the optimizer build speed in consistent directions and smooth out erratic, noisy zigzags caused by mini-batch training.
* Controlled by hyperparameter $\beta_1$ (usually set to $0.9$).


2. **Second Moment ($v_t$ - The Uncentered Variance / RMSprop):**
* This tracks the *magnitude* or spread of past squared gradients.
* If a parameter receives massive, violent gradient updates, $v_t$ grows large, causing Adam to shrink its effective learning rate for that specific parameter. If gradients are tiny, $v_t$ stays small, boosting the effective learning rate. This gives every parameter its own custom learning rate.
* Controlled by hyperparameter $\beta_2$ (usually set to $0.999$).



#### Bias Correction

Because both $m_t$ and $v_t$ are initialized at zero, they tend to be heavily biased toward zero during the initial steps of training (especially when $\beta_1$ and $\beta_2$ are close to $1$). Adam includes a clever **bias correction** step that scales up these estimates early in training so the network doesn't take sluggish steps at epoch 1.

---

### What Makes AdamW Different? (The True Standard)

While standard Adam was a revolution when it was introduced in 2014, researchers eventually discovered a flaw in how it handled **L2 Regularization** (weight decay, which prevents overfitting by penalizing large weights).

* In **Standard Adam**, L2 regularization was lumped directly into the gradient calculation. This meant parameters with large historical gradients received *less* regularization penalty than parameters with small historical gradients—an unintended side effect.
* In **AdamW**, **weight decay is decoupled**. The regularization penalty is applied directly to the weights *after* the adaptive gradient step has occurred.

This seemingly small mathematical correction drastically improved generalization performance, making **AdamW** the absolute gold standard for training Transformers, diffusion models, and complex neural networks today.

---

### Why is it the Undisputed Heavyweight?

* **Set-and-Forget Convenience:** Unlike basic SGD, which requires meticulous manual tuning of learning rate schedules and momentum values to get good results, Adam/AdamW works remarkably well out-of-the-box with default settings.
* **Handles Sparse Data & Noisy Gradients:** Because it scales learning rates per parameter, it excels on complex datasets with sparse features or noisy gradients.
* **Fast Initial Convergence:** It drops the loss value down rapidly during the early stages of training.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

In [2]:
# 1. Generate Vehicle Telemetry Dataset (4 features, 3 classes)
X_raw, y_raw = make_classification(
    n_samples=2000, 
    n_features=4, 
    n_informative=4, 
    n_redundant=0, 
    n_classes=3, 
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

In [3]:
# 2. Preprocess and Standardize Features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

In [4]:
# 3. Define the MLP Architecture
class VehicleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(VehicleMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Linear(16, output_dim) # Outputs raw logits for CrossEntropyLoss
        )
        
    def forward(self, x):
        return self.network(x)

model = VehicleMLP(input_dim=4, hidden_dim=32, output_dim=3)
criterion = nn.CrossEntropyLoss()

In [5]:
# 4. Initialize the AdamW Optimizer
# Notice the explicit weight_decay parameter, which applies proper L2 regularization
optimizer = optim.AdamW(
    model.parameters(), 
    lr=0.01,            # Learning rate
    weight_decay=0.01,  # Penalizes large, overly complex weights to prevent overfitting
    betas=(0.9, 0.999)  # Default momentum and uncentered variance decay rates
)

In [6]:
# 5. Training Loop
epochs = 100
print("--- Training MLP with AdamW ---")
for epoch in range(epochs):
    model.train()
    
    # Forward pass
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Backward pass & optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f}")

--- Training MLP with AdamW ---
Epoch [20/100] | Loss: 0.6748
Epoch [40/100] | Loss: 0.5140
Epoch [60/100] | Loss: 0.4449
Epoch [80/100] | Loss: 0.3739
Epoch [100/100] | Loss: 0.3419


In [7]:
# 6. Evaluation on Test Set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, predicted = torch.max(test_outputs, 1)
    accuracy = (predicted == y_test_t).sum().item() / y_test_t.size(0)
    print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")


Final Test Accuracy: 87.00%


Combining the **AdamW optimizer** with a **Cosine Annealing learning rate scheduler** is one of the most powerful and widely used recipes in modern deep learning (from training standard MLPs to massive Large Language Models).

While AdamW manages *how* each parameter updates based on gradients and weight decay, the scheduler manages *the overall step size* (learning rate) as training progresses.

---

### Why Cosine Annealing?

During the early stages of training, a **higher learning rate** helps the model make rapid progress and escape shallow local minima. However, as training nears completion, a high learning rate causes the weights to bounce around erratically, preventing the model from settling into the sharpest, most accurate minimum.

**Cosine Annealing** solves this by smoothly decaying the learning rate following the shape of a cosine wave:

* It starts high, drops off very slowly in the middle of training, accelerates its decay toward the end, and flattens out at a minimum learning rate.

Mathematically, the learning rate $\eta_t$ at epoch $t$ is calculated as:


$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{T_{\text{cur}}}{T_{\max}}\pi\right)\right)$$

* $\eta_{\max}$ & $\eta_{\min}$: Starting and minimum learning rates.
* $T_{\text{cur}}$: Current epoch number.
* $T_{\max}$: Total number of epochs for a half-cosine cycle.

---

### PyTorch Implementation: AdamW + Cosine Annealing

Here is how you integrate `torch.optim.lr_scheduler.CosineAnnealingLR` directly into our vehicle classification training loop:

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [9]:
# 1. Generate & Preprocess Data
X_raw, y_raw = make_classification(n_samples=2000, n_features=4, n_classes=3, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

ValueError: n_classes(3) * n_clusters_per_class(2) must be smaller or equal 2**n_informative(2)=4

In [ ]:
# 2. Define MLP Model
class VehicleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 3)
        )
    def forward(self, x):
        return self.network(x)

model = VehicleMLP()
criterion = nn.CrossEntropyLoss()

In [ ]:
# 3. Initialize AdamW Optimizer
epochs = 100
optimizer = optim.AdamW(
    model.parameters(), 
    lr=0.02,            # Initial peak learning rate (eta_max)
    weight_decay=0.01   # Decoupled weight decay
)



In [ ]:
# 4. Initialize Cosine Annealing Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=epochs,       # Number of epochs for the learning rate to decay to eta_min
    eta_min=1e-4        # The lowest learning rate at the very end of training
)

In [ ]:
# 5. Training Loop with Scheduler Step
print("--- Training MLP with AdamW & Cosine Annealing ---")
for epoch in range(epochs):
    model.train()
    
    # Forward pass & loss calculation
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    
    # Backward pass & optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Step the scheduler at the end of each epoch
    scheduler.step()
    
    # Print progress and current learning rate
    if (epoch + 1) % 20 == 0:
        current_lr = scheduler.get_last_lr()[0]
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Current LR: {current_lr:.6f}")

---

### Key Things to Remember

1. **`scheduler.step()` placement:** In PyTorch, epoch-based schedulers like `CosineAnnealingLR` should be called **once per epoch** (after your training loop finishes the epoch batches), whereas optimizer `.step()` is usually called per batch.
2. **The `T_max` parameter:** This tells the scheduler how long the cosine curve should take to reach its minimum value (`eta_min`). If your training runs longer than `T_max`, PyTorch will begin repeating or cycling the cosine curve depending on configuration.